In [0]:
# Databricks notebook source
# =========================================================================
# CÉLULA 2A: GERAÇÃO E FORMATAÇÃO DAS MENSAGENS (INDEPENDENTE DO SLACK)
# =========================================================================

import json
from pyspark.sql import functions as F

# 1. Carregar tabela consolidada de Lead Score
df_pipeline = spark.table("workspace.default.lead_scoring_pipeline")

# 2. Filtrar oportunidades ativas de Alta e Média Prioridade
df_priority_deals = df_pipeline.filter(F.col("priority_tier").isin("High Priority", "Medium Priority")) \
                               .orderBy(F.col("lead_score").desc())

# Converter para DataFrame Pandas para iterar e construir os payloads
deals_data = df_priority_deals.toPandas()

# 3. Função pura de montagem de payload em Block Kit do Slack
def build_slack_payload(deal):
    """
    Constrói o dicionário com os blocos visuais do Slack e o texto de explicabilidade.
    Não realiza nenhuma requisição externa.
    """
    tier_emoji = "🔥" if deal['priority_tier'] == "High Priority" else "⚡"
    
    payload = {
        "opportunity_id": deal['opportunity_id'],
        "sales_agent": deal['sales_agent'],
        "priority_tier": deal['priority_tier'],
        "lead_score": deal['lead_score'],
        "slack_block_kit": {
            "blocks": [
                {
                    "type": "header",
                    "text": {
                        "type": "plain_text",
                        "text": f"{tier_emoji} Novo Lead Prioritário — Score: {deal['lead_score']}/100",
                        "emoji": True
                    }
                },
                {
                    "type": "section",
                    "fields": [
                        {"type": "mrkdwn", "text": f"*Oportunidade:* {deal['opportunity_id']}"},
                        {"type": "mrkdwn", "text": f"*Cliente:* {deal['account']}"},
                        {"type": "mrkdwn", "text": f"*Vendedor:* {deal['sales_agent']}"},
                        {"type": "mrkdwn", "text": f"*Produto:* {deal['product']} (${deal['sales_price']:,})"}
                    ]
                },
                {"type": "divider"},
                {
                    "type": "section",
                    "text": {
                        "type": "mrkdwn",
                        "text": (
                            f"*Por que focar neste deal? (Explicabilidade)*\n"
                            f"🟢 *Ponto Forte:* {deal['reason_positive']}\n"
                            f"🔴 *Atenção/Risco:* {deal['reason_negative']}\n"
                            f"🕒 *Dias em Aberto:* {deal['days_in_pipeline']} dias"
                        )
                    }
                },
                {
                    "type": "section",
                    "text": {
                        "type": "mrkdwn",
                        "text": f"🎯 *Ação Recomendada:* `{deal['recommended_action']}`"
                    }
                },
                {"type": "divider"}
            ]
        }
    }
    return payload

# 4. Gerar lista de payloads em memória para os Top Deals de Alta Prioridade
top_deals = deals_data[deals_data['priority_tier'] == "High Priority"].head(5)

messages_to_send = []
for index, deal in top_deals.iterrows():
    messages_to_send.append(build_slack_payload(deal))

print(f"✅ Geração de mensagens concluída com sucesso!")
print(f"Total de {len(messages_to_send)} mensagens formatadas e salvas na memória (variável 'messages_to_send').\n")

# Exibir prévia da primeira mensagem gerada em formato JSON
if messages_to_send:
    print("--- Exemplo do Payload Gerado (Primeiro Lead) ---")
    print(json.dumps(messages_to_send[0]['slack_block_kit'], indent=2, ensure_ascii=False))

✅ Geração de mensagens concluída com sucesso!
Total de 5 mensagens formatadas e salvas na memória (variável 'messages_to_send').

--- Exemplo do Payload Gerado (Primeiro Lead) ---
{
  "blocks": [
    {
      "type": "header",
      "text": {
        "type": "plain_text",
        "text": "🔥 Novo Lead Prioritário — Score: 90/100",
        "emoji": true
      }
    },
    {
      "type": "section",
      "fields": [
        {
          "type": "mrkdwn",
          "text": "*Oportunidade:* L8XNQONA"
        },
        {
          "type": "mrkdwn",
          "text": "*Cliente:* Inity"
        },
        {
          "type": "mrkdwn",
          "text": "*Vendedor:* Zane Levy"
        },
        {
          "type": "mrkdwn",
          "text": "*Produto:* GTX Basic ($550.0)"
        }
      ]
    },
    {
      "type": "divider"
    },
    {
      "type": "section",
      "text": {
        "type": "mrkdwn",
        "text": "*Por que focar neste deal? (Explicabilidade)*\n🟢 *Ponto Forte:* Setor co

In [0]:
# Databricks notebook source
# =========================================================================
# CÉLULA 2B: ENVIO DAS MENSAGENS PARA O SLACK (EXECUTÁVEL OPCIONALMENTE)
# =========================================================================

import requests

# Insira a URL do Webhook do Slack. Deixe como string vazia ou desconfigurada se não for enviar.
SLACK_WEBHOOK_URL = "" 

# Validação prévia de credencial e lista de mensagens
if not SLACK_WEBHOOK_URL or "hooks.slack.com" not in SLACK_WEBHOOK_URL:
    print("⚠️ SLACK_WEBHOOK_URL não configurada ou inválida.")
    print("As mensagens foram geradas com sucesso na célula anterior, mas o envio HTTP foi pulado.")
elif 'messages_to_send' not in locals() or not messages_to_send:
    print("⚠️ Nenhuma mensagem encontrada na memória para envio. Execute primeiro a célula 2A.")
else:
    print(f"Iniciando envio de {len(messages_to_send)} notificações para o Slack...\n")
    
    success_count = 0
    for item in messages_to_send:
        response = requests.post(
            SLACK_WEBHOOK_URL,
            data=json.dumps(item['slack_block_kit']),
            headers={'Content-Type': 'application/json'}
        )
        
        if response.status_code == 200:
            print(f"✅ Notificação enviada para {item['sales_agent']} - Deal {item['opportunity_id']}")
            success_count += 1
        else:
            print(f"❌ Falha ao enviar deal {item['opportunity_id']} - HTTP Status: {response.status_code}")
            
    print(f"\nDisparo concluído: {success_count}/{len(messages_to_send)} notificações enviadas.")